# Locket stylize server on Colab

Runtime -> **Change runtime type -> T4 GPU**, then **Runtime -> Run all**. The last cell prints the
**address** and **token** for the app and then keeps running: leave it running while people use the app,
**Disconnect and delete runtime** when you are done.

One-time: put `server.tar` (made by `python colab/pack_code.py` on your PC) into Google Drive at `MyDrive/locket/`.
Optional: the key icon in the left bar -> add a secret `LOCKET_TOKEN` (fixed token, notebook access on).

In [ ]:
import subprocess
import urllib.request

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU -> Runtime -> Change runtime type -> T4 GPU, then run again")

In [ ]:
import os
import pathlib
import sys
import tarfile

from google.colab import drive

drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/locket")
DRIVE.mkdir(exist_ok=True)
WORK = pathlib.Path("/content")

assert (DRIVE / "server.tar").exists(), "Upload server.tar to Google Drive -> MyDrive/locket/ (see the first cell)"
tarfile.open(DRIVE / "server.tar").extractall(WORK, filter="data")      # always the latest code from Drive
print("code ready")

In [ ]:
# Colab runs Python 3.13, but the pinned libraries (numpy 1.26, mediapipe 0.10.21) stop at 3.12, so the
# server gets its own Python 3.12 environment with the same pins as server/requirements.txt.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
VENV = WORK / "venv"
PY = str(VENV / "bin" / "python")
if not VENV.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(VENV)], check=True)
skip = ("torch", "torchvision", "pytest", "httpx", "open_clip_torch")
pins = ["torch==2.6.0", "torchvision==0.21.0"]
for line in (WORK / "server" / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line and not line.lower().startswith(skip):
        pins.append(line)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PY, *pins], check=True)
print("libraries ready")

In [ ]:
import time

MODELS = WORK / "models"
WEIGHTS = {
    "RealESRGAN_x2plus.pth":
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    "blaze_face_short_range.tflite":
        "https://storage.googleapis.com/mediapipe-models/face_detector/"
        "blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
}

if not MODELS.exists():
    if (DRIVE / "models.tar").exists():
        # Drive is slow to read (a 6 GB file can take 5-15 min), and extractall prints nothing:
        # unpack member by member so the cell shows how far it is.
        print("restoring models from Drive (5-15 min, progress below)...")
        MODELS.mkdir()
        archive = DRIVE / "models.tar"
        total = archive.stat().st_size
        started = time.time()
        next_note = 0.05
        with tarfile.open(archive, "r:") as tar:
            for member in tar:
                tar.extract(member, MODELS, filter="data")
                done = tar.fileobj.tell() / total
                if done >= next_note:
                    print(f"  {done:.0%} restored ({int(time.time() - started)} s)", flush=True)
                    next_note = done + 0.05
        print("restored in", int(time.time() - started), "s")
    else:
        print("first run: downloading ~6 GB of models (~5-10 min), saved to Drive for next time")
        (MODELS / "weights").mkdir(parents=True)
        for name, url in WEIGHTS.items():
            urllib.request.urlretrieve(url, MODELS / "weights" / name)
        env = dict(os.environ, HF_HOME=str(MODELS / "hf"), TORCH_HOME=str(MODELS / "torch"))
        subprocess.run([PY, "download_models.py"], cwd=WORK / "server", env=env, check=True)
        print("saving the models to Drive for next time (a few minutes)...")
        with tarfile.open(DRIVE / "models.tar", "w") as tar:
            tar.add(MODELS / "hf", arcname="hf")
            tar.add(MODELS / "weights", arcname="weights")
if not (WORK / "server" / "weights").exists():
    os.symlink(MODELS / "weights", WORK / "server" / "weights")
print("models ready")

In [ ]:
import secrets

try:
    from google.colab import userdata

    TOKEN = userdata.get("LOCKET_TOKEN")
except Exception:
    TOKEN = secrets.token_urlsafe(24)
    print("No Colab secret LOCKET_TOKEN: using a random token for this session only")

In [ ]:
import json
import time

# MPLBACKEND: Colab points matplotlib at its notebook backend, which the server's own Python
# does not have; mediapipe imports matplotlib and would crash. The server has no screen anyway.
ENV = dict(os.environ, LOCKET_TOKEN=TOKEN, HF_HOME=str(MODELS / "hf"),
           TORCH_HOME=str(MODELS / "torch"), PYTHONWARNINGS="ignore", MPLBACKEND="Agg")


def start_server():
    return subprocess.Popen(
        [PY, "-m", "uvicorn", "locket_server.api:app_factory", "--factory",
         "--host", "127.0.0.1", "--port", "8765"],
        cwd=WORK / "server", env=ENV, stdout=open(WORK / "server.log", "a"), stderr=subprocess.STDOUT)


def health(base="http://127.0.0.1:8765"):
    try:
        return json.load(urllib.request.urlopen(base + "/v1/health", timeout=10))
    except Exception:
        return None


def log_tail(chars=2500):
    path = WORK / "server.log"
    return path.read_text(errors="replace")[-chars:] if path.exists() else "(no log yet)"


def wait_ready(proc, seconds=1500):
    deadline = time.time() + seconds
    last_note = time.time()
    while time.time() < deadline:
        info = health()
        if info and info.get("models_ready"):
            return info
        if info and info.get("warmup_error"):
            raise RuntimeError("models failed to load: " + info["warmup_error"] + "\n--- server.log ---\n" + log_tail())
        if proc.poll() is not None:
            raise RuntimeError("server exited:\n" + log_tail())
        if time.time() - last_note > 60:
            last_note = time.time()
            print(f"still loading models ({int(deadline - time.time())} s left)... last log line:",
                  log_tail(300).strip().splitlines()[-1:] or "-")
        time.sleep(5)
    raise TimeoutError("models did not load in time. Last log:\n" + log_tail())


server = start_server()
print(wait_ready(server))

In [ ]:
import re
import threading

CLOUDFLARED = WORK / "cloudflared"
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)


def start_tunnel():
    proc = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if found:
            break
    else:
        raise RuntimeError("cloudflared did not give an address")
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()  # keep the pipe drained
    url = found.group(0)
    for _ in range(30):                      # the new address needs a few seconds to resolve
        if health(url):
            break
        time.sleep(2)
    return proc, url


def show(url):
    print("=" * 60)
    print("  App -> camera screen -> server icon:")
    print("  Address:", url)
    print("  Token  :", TOKEN)
    print("=" * 60)


tunnel, URL = start_tunnel()
show(URL)

In [ ]:
# Keep this cell running while the app is in use. It restarts the server or the tunnel if either dies.
# A new tunnel means a NEW address: it is printed again, enter it in the app.
beat = 0
while True:
    time.sleep(60)
    beat += 1
    if server.poll() is not None:
        print("server stopped, restarting...")
        server = start_server()
        wait_ready(server)
    if tunnel.poll() is not None:
        print("tunnel stopped, opening a new one...")
        tunnel, URL = start_tunnel()
        show(URL)
    if beat % 10 == 0:
        info = health() or {}
        print(time.strftime("%H:%M"), "alive | gpu:", info.get("gpu"), "| free MB:", info.get("vram_free_mb"))